# 01 Data Cleanup

1.Import libraries

In [3]:
import pandas as pd
import re

2. Load the original data

In [2]:
file_path = "../data/raw/chatlog_translated_sampled.csv"

df = pd.read_csv(file_path)

df.head()

,userID,time,role,message,detected_ad_name,message_english
0,6736498766402264,2026-04-06 22:18:07,User,ใครมีหนี้เยอะ!! รวมยอดหนี้ทักแชทหาพี่ที่<ORG>ไ...,Photo_Messenger_BannerC4_APR26_Group_1,"Who has a lot of debt!! Total debt amount, cha..."
1,6736498766402264,2026-04-06 22:18:14,User,AdsMainmenu,NaN,AdsMainmenu
2,6736498766402264,2026-04-06 22:18:15,Admin,"{'type': 'quick_reply', 'list_object': [{'text...",NaN,"{'type': 'quick_reply', 'list_object': [{'text..."
3,6746342932062781,2026-04-04 17:06:16,User,Greeting,NaN,Greeting
4,6746342932062781,2026-04-04 17:06:18,Admin,"{'type': 'image', 'list_object': [{'original_u...",NaN,"{'type': 'image', 'list_object': [{'original_u..."


3. Understand the dataset

In [4]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.info()

Dataset shape: (22750, 6)

Columns:
['userID', 'time', 'role', 'message', 'detected_ad_name', 'message_english']
<class 'pandas.DataFrame'>
RangeIndex: 22750 entries, 0 to 22749
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   userID            22750 non-null  int64
 1   time              22750 non-null  str  
 2   role              22750 non-null  str  
 3   message           22750 non-null  str  
 4   detected_ad_name  986 non-null    str  
 5   message_english   22750 non-null  str  
dtypes: int64(1), str(5)
memory usage: 1.0 MB


4. Check missing values

In [5]:
df.isnull().sum()

userID                  0
time                    0
role                    0
message                 0
detected_ad_name    21764
message_english         0
dtype: int64

5. Check duplicates

In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 7


7 rows are duplicated. which means 14 rows

Remove exact duplicates:

In [9]:
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (22743, 6)


6. Clean role names: Remove spaces and standardizes capitalization

In [10]:
df["role"] = (
    df["role"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["role"].value_counts()

role
admin    12708
user     10035
Name: count, dtype: int64

7. Convert time to datetime

In [11]:
df["time"] = pd.to_datetime(df["time"], errors="coerce")

print("Invalid timestamps:", df["time"].isnull().sum())

Invalid timestamps: 0


8. Select customer messages: For customer-demand analysis, focus on messages where the role is user.

In [12]:
customer_df = df[df["role"] == "user"].copy()

print("Customer messages:", len(customer_df))

Customer messages: 10035


9. Identify advertisement-entry messages

In [13]:
customer_df["is_ad_entry"] = customer_df["detected_ad_name"].notna()

customer_df["is_ad_entry"].value_counts()

is_ad_entry
False    9051
True      984
Name: count, dtype: int64

Separate genuine customer messages from advertisement entries:

In [14]:
keyword_df = customer_df[
    customer_df["is_ad_entry"] == False
].copy()

10. Normalize English text

In [15]:
def clean_text(text):
    text = str(text).lower().strip()

    # Remove anonymized personal information
    text = re.sub(r"<(?:name|phone|link|org)>", " ", text)

    # Remove chatbot formatting symbols
    text = text.replace("[[", " ").replace("]]", " ")
    text = text.replace("{{", " ").replace("}}", " ")

    # Keep letters, numbers, apostrophes and hyphens
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)

    # Replace repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

Apply it:

In [16]:
keyword_df["clean_text"] = keyword_df["message_english"].apply(clean_text)

keyword_df[["message_english", "clean_text"]].head(10)

,message_english,clean_text
1,AdsMainmenu,adsmainmenu
3,Greeting,greeting
7,Debt consolidation project,debt consolidation project
9,What kind?,what kind
11,Loan details,loan details
13,Motorcycle loan or refinance,motorcycle loan or refinance
16,[[Refinance]]Refinance motorcycle,refinance refinance motorcycle
23,Not yet.,not yet
26,correct,correct
28,"If I get a loan, can I use it to cover Srisawat?",if i get a loan can i use it to cover srisawat


11. Remove empty and personal-information-only messages

In [17]:
keyword_df = keyword_df[
    keyword_df["clean_text"] != ""
].copy()

12. Flag low-information responses

In [18]:
low_information = {
    "yes",
    "no",
    "ok",
    "okay",
    "correct",
    "agree",
    "edit",
    "greeting",
    "adsmainmenu"
}

keyword_df["is_low_information"] = (
    keyword_df["clean_text"].isin(low_information)
)

keyword_df["is_low_information"].value_counts()

is_low_information
False    6525
True     1515
Name: count, dtype: int64

Remove them from keyword analysis:

In [19]:
keyword_df = keyword_df[
    keyword_df["is_low_information"] == False
].copy()

13. Review the result

In [20]:
print("Original rows:", len(df))
print("Customer messages:", len(customer_df))
print("Keyword-ready messages:", len(keyword_df))

keyword_df[
    ["userID", "time", "message_english", "clean_text"]
].sample(10, random_state=42)

Original rows: 22743
Customer messages: 10035
Keyword-ready messages: 6525


,userID,time,message_english,clean_text
16785,2737469059659633,2026-06-06 10:32:53,automatic transmission,automatic transmission
19673,25721231704213579,2026-06-22 18:23:18,Please inquire. \nPlease inquire.\nHow many da...,please inquire please inquire how many days af...
21436,27353287154329071,2026-06-22 13:57:39,Car loan link,car loan link
7951,27157299963871830,2026-04-21 15:24:28,Want to consolidate debt,want to consolidate debt
20451,27341853552102048,2026-06-19 09:31:06,I'm interested in a loan by calling.,i'm interested in a loan by calling
10957,25787344450917012,2026-05-15 12:35:12,"Yes, wait a moment.",yes wait a moment
18981,2388051857989091,2026-06-15 17:56:55,Total debt,total debt
18039,7971034532967640,2026-06-03 15:34:31,"Yes, yes.",yes yes
8137,7405779136102132,2026-05-11 14:24:09,I would like to ask if I would like to close t...,i would like to ask if i would like to close t...
16257,9876420645735319,2026-06-13 15:33:11,Car insurance link,car insurance link


14. Save the cleaned data

In [21]:
output_path = "../data/processed/keyword_ready_messages.csv"

keyword_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: ../data/processed/keyword_ready_messages.csv


In [22]:
original_rows, original_columns = df.shape
cleaned_rows, cleaned_columns = keyword_df.shape

print(f"Original dataset: {original_rows:,} rows and {original_columns} columns")
print(f"Cleaned dataset: {cleaned_rows:,} rows and {cleaned_columns} columns")
print(f"Rows removed: {original_rows - cleaned_rows:,}")

Original dataset: 22,743 rows and 6 columns
Cleaned dataset: 6,525 rows and 9 columns
Rows removed: 16,218
